## File contents

#### Setup (top of file)

Imports helper functions and config.   

    1. Builds catchments_with_flood_output:   
    2. Checks, for each catchment, whether the Ens01 10 cm flooded-area file exists, and keeps those catchments.

#### Then, for each catchment:
    3. Skips it if the combined results file (..._new_Oct.pkl) already exists.
    4. Loads the catchment boundary polygon.
    5. Loads a tiny rainfall cube purely to build the grid mask: FULL_MASK_2D (which cells of the full grid fall inside the catchment) and mask_2d_sub (the same mask cropped to the bounding box).

#### For each ensemble member:
    6. If a per-member pickle exists, loads it and moves on.
    7. Otherwise reads the events CSV, numbers the events 1 to N and assigns each a hydro year (December events belong to the following year).
    8. Builds event_details_cache, a dictionary with each event’s start and stop indices, times, duration and CSV peak.

#### For each hydro year:
    9. Loads the daily soil moisture (SM) file for 1 Dec to 30 Nov, plus the previous year’s file if an event starts on 1 December. It masks and crops the data to the catchment and builds sm_day_idx, a date to array-index lookup.

#### For each event:
    10. Loads the hourly rainfall for the event window (starting one hour early), masks it and crops it to the bounding box.
    11. Finds the grid cell and hour of peak rainfall, and checks the cube’s time range contains the CSV start time.
    12. Adds the event metadata, the peak date, day-of-year in two forms (hydro_day_360, day_360) and a check that the cube’s peak matches the CSV’s.
    13. Takes the hourly series at the peak cell and finds the first and last wet hours (above 0.1), giving the onset date (anchor_key).
    14. Loads the flood arrays (10 and 30 cm, area and volume) once per ensemble member, then reads flood values at the peak cell for this event.
    15. Takes SM at that cell for the day before the onset.
    16. Runs analyse_peak_event for the spatial neighbourhood and threshold analysis, and appends the event’s dictionary to the results.

#### Saving:
    17. Writes a per-member pickle, concatenates all members into the combined pickle, and deletes the per-member files.

In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

from HelperFunctions import *  
from ProcessingFunctions import (load_flood_arrays, load_rainfall_events, add_hydro_year, find_wet_onset, sm_day_before,
                                  build_catchment_mask, load_soil_moisture,extract_at_cell, get_rainfall_data_from_climate_data,
                                  get_rainfall_at_peak_cells, find_peak_cell, add_rainfall_peak_time_vars, 
                                 find_spatial_stats_around_peak)

# ── Config ────────────────────────────────────────────────────────────────────
from config import RAINFALL_CSV_DIR, CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS, ENSEMBLE_MEMBERS 

FLOOD_ROOT = "/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/"
RAIN_ROOT = "/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/"
RAIN_CUBE_ROOT = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/"
SM_ROOT =   '/scratch/hydro4/shared_data/climate_projections/UKCP18/UKCP_local/Soil_moisture/5km_regridded/'

WET_THRESH   = 0.1          # mm/hr; an hour counts as wet above this
FLOOD_DEPTHS = [10, 30]     # cm

full_rain_cube = get_rainfall_cube_subsection(2015, '01', f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/01/", 1, 2)

## ------------------------------------------------------------ ###
## Define which catchments to run
## ------------------------------------------------------------ ###

In [2]:
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())
catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = RAIN_ROOT + f"Catchment_{catchment_num}/{catchment_name}.pkl"
    flood_fp = FLOOD_ROOT + f"Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp):
        catchments_with_flood_output.append(catchment_num)    

## ------------------------------------------------------------ ###
## Loop through catchments processing
## ------------------------------------------------------------ ###

In [ ]:
for catchment_num in ['33_b']: # catchments_with_flood_output:
     
    # Make directory for outputs, if it doesn't already exist
    print(f"Running for {catchment_num}")
    if not os.path.exists(os.path.join(OUT_DIR, f"Catchment_{catchment_num}")):
        os.mkdir(os.path.join(OUT_DIR, f"Catchment_{catchment_num}"))

    # If output already exists, then skip this catchment
    catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]  
    catchment_fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_new_Nov.pkl")
    print(catchment_fp)
    if os.path.exists(catchment_fp):
        print("Already exists")
        continue        
        
    # Get supporting spatial data and create mask 
    boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
    CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]         
    mask, mask_sub = build_catchment_mask(full_rain_cube, CATCHMENT_POLY)

    # Create structure to store results for whole catchment
    this_catchment_all_results = []
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────
    #    Loop through each ensemble member for this catchment
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────    
    
    for ens_num in tqdm(ENSEMBLE_MEMBERS):
        print(f"Running for EM {ens_num}")
        member_fp = os.path.join(OUT_DIR,f"Catchment_{catchment_num}",f"{catchment_name}_EM{ens_num}_new_Nov.pkl")
        
        # If output already exists, then skip this ensemble member
        if os.path.exists(member_fp):
            this_catchment_all_results.append(pd.read_pickle(member_fp)); continue

        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        # Get rainfall event data, add variables AND get flood data
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        events = add_hydro_year(load_rainfall_events(RAINFALL_CSV_DIR, catchment_name, ens_num))
        flood_cubes, flood_numpy = load_flood_arrays(FLOOD_ROOT, catchment_num, ens_num, mask, CATCHMENT_POLY)   
        
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        # Create structure to store results
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        this_ens_all_results = []
        
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        # Loop through each year and all the events in that year
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────        
        for hydro_year, events_in_year in tqdm(events.groupby('hydro_year')):
            
            # ─────────────────────────────────────────────────────────────────────────────────────────────────────
            # Get soil moisture data for this year  
            # ─────────────────────────────────────────────────────────────────────────────────────────────────────
            sm = load_soil_moisture(SM_ROOT, events_in_year, ens_num, hydro_year, CATCHMENT_POLY, mask)

            # ─────────────────────────────────────────────────────────────────────────────────────────────────────
            # Loop through all events within this 'hydro-year'
            # ─────────────────────────────────────────────────────────────────────────────────────────────────────
            
            for row in events_in_year.itertuples():
                
                # Get event number
                event_idx = int(row.event_num) - 1
                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #                  Create results dictionary for this event with basic details 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                
                result = {'ens': ens_num, 'hydro_year': hydro_year, 'event_num': int(row.event_num),
                        'start_idx': int(row.start_indices), 'stop_idx': int(row.stop_indices),
                        'max_precip_from_csv': float(row.peaks),
                        'start_year': int(row.start_year), 'start_month': int(row.start_month),
                        'start_day': int(row.start_day),   'start_hour': int(row.start_hour),
                        'event_durations': int(row.event_durations)}

                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #                Get rainfall data for the hydro-year this event happened in 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────

                rainfall_year_cube, time_coord, time_mismatch = get_rainfall_data_from_climate_data(RAIN_CUBE_ROOT, hydro_year,
                                                                                                    ens_num, row, mask, CATCHMENT_POLY)
                result['time_mismatch'] = time_mismatch

                # Subset to catchment and get spatial offsets (so that cell references for within the catchment can also
                # be used on data that hasn't been subset to the catchment)
                rainfall_year_cube, x_offset, y_offset = subset_cube_to_bbox(rainfall_year_cube, CATCHMENT_POLY, buffer=0)

                # Check flood, rainfall and soil moisture data are all the same size
                # assert flood_numpy[10]['vol'].shape[1:] == rainfall_year_cube.shape[1:]  == sm.data.shape[1:]

                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #           Find the cell with the peak value in both rainfall and flooding data 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────

                rain_pk  = find_peak_cell(rainfall_year_cube.data)  # (t, y, x)
                flood_pk = find_peak_cell(flood_numpy[10]['area'][event_idx])     # (y, x)

                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #   Extract the rainfall data at both of those locations
                #       - Record whether the two peaks were recorded at same location, and how far apart they are
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────

                result = get_rainfall_at_peak_cells (rain_pk, "rainpeak", rainfall_year_cube, flood_numpy, sm,  x_offset, y_offset, 
                                                     event_idx, WET_THRESH, result)
                result = get_rainfall_at_peak_cells (flood_pk, "floodpeak", rainfall_year_cube, flood_numpy, sm,  x_offset, y_offset,
                                                     event_idx, WET_THRESH, result)

                if rain_pk and flood_pk:
                    result['peaks_same_cell'] = (rain_pk['y_idx'], rain_pk['x_idx']) == (flood_pk['y_idx'], flood_pk['x_idx'])
                    # Find the distance (using the hypotenuse) between the two peak cells
                    result['peaks_dist_cells'] = float(np.hypot(rain_pk['y_idx'] - flood_pk['y_idx'],
                                                                rain_pk['x_idx'] - flood_pk['x_idx']))    
                else:
                    result['peaks_same_cell'] =np.nan
                    result['peaks_dist_cells'] = np.nan

                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #   Record the time at which the rainfall peak occurred in the rainfall data
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────            
                result = add_rainfall_peak_time_vars(time_coord, result)

                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #   Record the time at which the rainfall peak occurred in the rainfall data
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
                result = find_spatial_stats_around_peak(result, rainfall_year_cube, flood_numpy, flood_cubes, 
                                         CATCHMENT_POLY,boundary_gdf, 1, [0.5, 0.6, 0.8], plot=False)
                
                result_old=pd.DataFrame(result)
                result = pd.DataFrame([result])
                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────
                #   Add to ensemble member results, and catchment results
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
                # ─────────────────────────────────────────────────────────────────────────────────────────────────────  
                # print(row.peaks, result['rainpeak_max_precip'])

                this_ens_all_results.append(result)
                this_catchment_all_results.append(result)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────
        #   Save results for this ensemble member
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
        # ─────────────────────────────────────────────────────────────────────────────────────────────────────                 
        this_ens_all_results_df = pd.concat(this_ens_all_results, ignore_index=True).to_pickle(member_fp)
        
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────
    #   Save results for this catchment
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────   
    # ─────────────────────────────────────────────────────────────────────────────────────────────────────    
    this_catchment_all_results_df = pd.concat(this_catchment_all_results, ignore_index=True) 
    this_catchment_all_results_df.to_pickle(catchment_fp)    

    # Delete individual ensemble member results
    for ens_num in ENSEMBLE_MEMBERS:
        member_fp = os.path.join(OUT_DIR,f"Catchment_{catchment_num}",f"{catchment_name}_EM{ens_num}_new_Nov.pkl")
        if os.path.exists(member_fp):
            os.remove(member_fp)

Running for 33_b
/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/Catchment_33_b/GreatOuse_b_new_Nov.pkl


  0%|                                                                                                                                           | 0/12 [00:00<?, ?it/s]

Running for EM 01
Running for EM 04


 17%|█████████████████████▊                                                                                                             | 2/12 [00:00<00:01,  8.76it/s]

Running for EM 05
Running for EM 06
Running for EM 07
Running for EM 08
Running for EM 09



 87%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                 | 46/53 [06:18<00:47,  6.76s/it]


 58%|████████████████████████████████████████████████████████████████████████████▍                                                      | 7/12 [07:30<05:51, 70.35s/it]

Running for EM 10



 21%|███████████████████████████▎                                                                                                      | 13/62 [01:32<05:37,  6.90s/it]

### Make compareson with old results

In [5]:
catchment_num = '19'
catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
old = pd.read_pickle(OUT_DIR + "Catchment_23/Tyne(Northumberland)_new.pkl")
new = pd.read_pickle(OUT_DIR + "Catchment_23/Tyne(Northumberland)_new_Nov.pkl")

In [ ]:
# new['rainpeak_sm_day_before_onset'], old['mean_sm_2_before_event']

In [6]:
# tolerances for columns that are now rounded when stored
atol = {'max_precip': 0.05, '10cm_area': 0.005, '30cm_area': 0.005,
        '10cm_volume': 1.0, '30cm_volume': 1.0}

rename = {   # old name -> new name (rainfall-peak anchor; floodpeak_* has no old equivalent)
    'x_idx': 'rainpeak_x_idx', 'y_idx': 'rainpeak_y_idx',
    'x_idx_global': 'rainpeak_x_global', 'y_idx_global': 'rainpeak_y_global',
    'max_precip': 'rainpeak_peak_value', 't_local': 'rainpeak_t_local',
    '10cm_area': 'rainpeak_10cm_area',  '10cm_volume': 'rainpeak_10cm_volume',
    '30cm_area': 'rainpeak_30cm_area',  '30cm_volume': 'rainpeak_30cm_volume',
    'hydro_day_360': 'rainfall_peak_day_hydro_360day',
    'day_360': 'rainfall_peak_day_360day',
}
same = ['start_idx', 'stop_idx', 'start_year', 'start_month', 'start_day', 'start_hour',
        'event_durations', 'max_precip_from_csv', 'time_mismatch', 't_global', 'rainfall_peak_day']

# analysis columns: same name in old and new
analysis_cols = [c for c in old.columns if c.startswith(('neighbourhood_', 'threshold_', 'cluster_'))]

col_map = {**{c: c for c in same + analysis_cols}, **rename}

# Decimal places each NEW column is rounded to when stored (missing key = not rounded)
decimals = {
    'rainpeak_peak_value': 1,
    'rainpeak_10cm_area': 2, 'rainpeak_30cm_area': 2,
    'rainpeak_10cm_volume': 1, 'rainpeak_30cm_volume': 1,
}
decimals.update({c: 1 for c in analysis_cols if 'n_cells' not in c})   # analyse_peak_event rounds these to 1 dp

rows = []
for o, n in col_map.items():
    if o not in old.columns or n not in new.columns:
        rows.append({'old': o, 'new': n, 'status': 'MISSING'}); continue
    a, b = old[o], new[n]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        a, b = a.astype(float), b.astype(float)
        ok = np.isclose(a, b, rtol=1e-6, atol=atol.get(o, 1e-6), equal_nan=True)
        maxdiff = np.nanmax(np.abs(a - b)) if (~ok).any() else 0.0
    else:
        ok = ((a == b) | (a.isna() & b.isna())).to_numpy()
        maxdiff = np.nan
    rows.append({'old': o, 'new': n, 'status': 'ok' if ok.all() else 'DIFF',
                 'n_diff': int((~ok).sum()), 'max_abs_diff': maxdiff,
                 'first_events': list(a.index[~ok][:5])})

report = pd.DataFrame(rows)
print(report.sort_values('n_diff', ascending=False).to_string())

                                   old                                new   status  n_diff  max_abs_diff     first_events
10                   rainfall_peak_day                  rainfall_peak_day     DIFF   867.0           NaN  [0, 1, 2, 4, 5]
0                            start_idx                          start_idx       ok     0.0           0.0               []
1                             stop_idx                           stop_idx       ok     0.0           0.0               []
2                           start_year                         start_year       ok     0.0           0.0               []
4                            start_day                          start_day       ok     0.0           0.0               []
3                          start_month                        start_month       ok     0.0           0.0               []
6                      event_durations                    event_durations       ok     0.0           0.0               []
7                  max_p

### Double check if there are any other diverging columns

In [ ]:
rename_new_to_old = {   # old name -> new name (rainfall-peak anchor; floodpeak_* has no old equivalent)
    'x_idx': 'rainpeak_x_idx', 'y_idx': 'rainpeak_y_idx',
    'x_idx_global': 'rainpeak_x_global', 'y_idx_global': 'rainpeak_y_global',
    'max_precip': 'rainpeak_peak_value', 't_local': 'rainpeak_t_local',
    '10cm_area': 'rainpeak_10cm_area',  '10cm_volume': 'rainpeak_10cm_volume',
    '30cm_area': 'rainpeak_30cm_area',  '30cm_volume': 'rainpeak_30cm_volume',
    'hydro_day_360': 'rainfall_peak_day_hydro_360day',
    'day_360': 'rainfall_peak_day_360day', 'x_coord': 'rainpeak_x_coord' , 'y_coord': 'rainpeak_y_coord',
    "total_acc":  'rainpeak_total_precip', 'values': "rainpeak_rain_series"
}

rename_new_to_old = {v: k for k, v in rename_new_to_old.items()}


new = new.rename(columns=rename_new_to_old)

In [ ]:
old_cols = set(old.columns)
new_cols = set(new.columns)

print("Only in old:")
print(sorted(old_cols - new_cols))

print("\nOnly in new:")
print(sorted(new_cols - old_cols))

print("\nIn both:")
print(sorted(old_cols & new_cols))